# 📘 Agentic 架构 5：Multi-Agent Systems（多智能体系统）

本 notebook 进入最强大、最灵活的架构之一：**Multi-Agent System（多智能体系统）**。它不再局限于单个 agent（无论多复杂），而是建模**协作求解问题的专业 agent 团队**；每个 agent 有明确角色、人设与技能，类似人类专家团队。

这种方式实现深刻的「劳动分工」：复杂问题被拆成子任务，交给最合适的 agent。为展示效果，我们做直接对比：先让单一的**单体「通才」agent**撰写完整市场分析报告；再组建**专业团队**——技术分析师、新闻分析师、财务分析师——并由第四位「管理者」agent 综合专家输出形成终稿。这两种方式在质量、结构与深度的差异会非常明显。

### 定义
**Multi-Agent System** 指多个不同、专业化的 agent 为共同目标协作（或有时竞争）的架构。通过中央控制器或约定的工作流协议管理通信与任务路由。

### 高层工作流

1. **分解：** 主控制器或用户提供复杂任务。
2. **角色定义：** 按预设角色（如 Researcher、Coder、Critic、Writer）将子任务分给各 agent。
3. **协作：** agent 并行或顺序执行任务，彼此传递输出，或写入中央「黑板」。
4. **综合：** 最终由「管理者」或「综合者」agent 收集专家输出并组装成统一回复。

### 适用场景 / 应用
* **复杂报告：** 需金融、科研等多领域专长的详细报告。
* **软件开发流水线：** 模拟开发、评审、测试、项目经理等角色。
* **创意头脑风暴：** 不同「性格」的 agent（乐观、谨慎、天马行空）可产生更多样想法。

### 优势与局限
* **优势：**
    * **专业化与深度：** 每个 agent 可针对人设与工具微调，领域内质量更高。
    * **模块化与可扩展：** 增删、升级单个 agent 不必重做整个系统。
    * **并行：** 多 agent 可同时处理子任务，可能缩短总耗时。
* **局限：**
    * **协调成本：** 管理 agent 间通信与工作流增加设计复杂度。
    * **成本与延迟：** 多轮 LLM 调用通常比单 agent 更贵、更慢。

## 阶段 0：基础与环境

安装库，并为 DeepSeek、LangSmith 与 Tavily 配置 API 密钥。

### 步骤 0.1：安装核心库

**本节将做什么：**
安装本系列项目常用的标准库套件。

In [1]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv langchain-tavily

### 步骤 0.2：导入库并配置密钥

**本节将做什么：**
导入模块并从 `.env` 加载 API 密钥。

**需要操作：** 在本目录创建 `.env` 并填入密钥：
```
DEEPSEEK_API_KEY="your_deepseek_api_key_here"
LANGCHAIN_API_KEY="your_langsmith_api_key_here"
TAVILY_API_KEY="your_tavily_api_key_here"
```

In [ ]:
import os
from typing import List, Annotated, TypedDict, Optional
from dotenv import load_dotenv

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from langgraph.graph.message import AnyMessage, add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

# --- API Key and Tracing Setup ---
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Multi-Agent (DeepSeek)"

for key in ["DEEPSEEK_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：基线——单体「通才」agent

要体现专业团队的价值，先看单个 agent 在复杂任务上的表现。我们构建 ReAct agent，并用宽泛提示要求其同时做多类分析。

### 步骤 1.1：构建单体 agent

**本节将做什么：**
构建标准 ReAct agent，提供网页搜索工具，以及要求其担任全面金融分析师的通用系统提示。

In [3]:
console = Console()

# Define the shared state for both agents
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

# Define the tool and LLM
search_tool = TavilySearch(max_results=3, name="web_search")
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
    temperature=0,
)
llm_with_tools = llm.bind_tools([search_tool])

# Define the monolithic agent node
def monolithic_agent_node(state: AgentState):
    console.print("--- MONOLITHIC AGENT: Thinking... ---")
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode([search_tool])

# Build the ReAct graph for the monolithic agent
mono_graph_builder = StateGraph(AgentState)
mono_graph_builder.add_node("agent", monolithic_agent_node)
mono_graph_builder.add_node("tools", tool_node)
mono_graph_builder.set_entry_point("agent")

def tools_condition_with_end(state):
    result = tools_condition(state)
    if isinstance(result, str):
        # Older versions return just "tools" or "agent"
        return {result: "tools", "__default__": END}
    elif isinstance(result, dict):
        # Newer versions return a mapping
        result["__default__"] = END
        return result
    else:
        raise TypeError(f"Unexpected type from tools_condition: {type(result)}")

mono_graph_builder.add_conditional_edges("agent", tools_condition_with_end)
mono_graph_builder.add_edge("tools", "agent")

monolithic_agent_app = mono_graph_builder.compile()

print("Monolithic 'generalist' agent compiled successfully.")

Monolithic 'generalist' agent compiled successfully.


### 步骤 1.2：测试单体 agent

**本节将做什么：**
给通才 agent 复杂任务：为某公司撰写完整市场分析报告，覆盖三个不同方面。

In [4]:
company = "NVIDIA (NVDA)"
monolithic_query = f"Create a brief but comprehensive market analysis report for {company}. The report should include three sections: 1. A summary of recent news and market sentiment. 2. A basic technical analysis of the stock's price trend. 3. A look at the company's recent financial performance."

console.print(f"[bold yellow]Testing MONOLITHIC agent on a multi-faceted task:[/bold yellow]\n'{monolithic_query}'\n")

final_mono_output = monolithic_agent_app.invoke({
    "messages": [
        SystemMessage(content="You are a single, expert financial analyst. You must create a comprehensive report covering all aspects of the user's request."),
        HumanMessage(content=monolithic_query)
    ]
})

console.print("\n--- [bold red]Final Report from Monolithic Agent[/bold red] ---")
console.print(Markdown(final_mono_output['messages'][-1].content))

Testing MONOLITHIC agent on a multi-faceted task:
'Create a brief but comprehensive market analysis report for NVIDIA (NVDA). The report should include three 
sections: 1. A summary of recent news and market sentiment. 2. A basic technical analysis of the stock's price 
trend. 3. A look at the company's recent financial performance.'

--- MONOLITHIC AGENT: Thinking... ---

Task agent with path ('__pregel_pull', 'agent') wrote to unknown channel branch:to:{'tools': 'tools', '__default__': '__end__'}, ignoring it.


--- Final Report from Monolithic Agent ---

**输出说明：**
单体 agent 会生成报告，可能进行了多次网页搜索并尽力综合。但输出常有不足：
- **结构不足：** 各节可能混在一起，缺少清晰标题或专业版式。
- **分析浅显：** 同时扮演三个领域的「专家」，可能在每一领域都只给高层概述。
- **语气泛化：** 语言可能缺乏各领域真正专家的行话与聚焦。

这是基线：能用，但不出彩。接下来组建专业团队看能否做得更好。

## 阶段 2：进阶方案——多智能体专业团队

组建团队：新闻分析师、技术分析师、财务分析师，各为独立 agent 节点并带专属人设。最后由 Report Writer 担任管理者，汇总工作。

### 步骤 2.1：定义专业 agent 节点

**本节将做什么：**
创建三个不同的 agent 节点。关键在于为每个节点提供**高度具体的系统提示**：定义人设、专长领域与输出格式，以此落实专业化。

In [ ]:
# The state for our multi-agent system will hold the outputs of each specialist
class MultiAgentState(TypedDict):
    user_request: str
    news_report: Optional[str]
    technical_report: Optional[str]
    financial_report: Optional[str]
    final_report: Optional[str]

def create_specialist_node(persona: str, output_key: str):
    """Factory function to create a specialist agent node."""
    system_prompt = persona + "\n\nYou have access to a web search tool. Your output MUST be a concise report section, formatted in markdown, focusing only on your area of expertise."

    # ✅ Build a ChatPromptTemplate instead of a plain list
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{user_request}")
    ])

    agent = prompt_template | llm_with_tools

    def specialist_node(state: MultiAgentState):
        console.print(f"--- CALLING {output_key.replace('_report','').upper()} ANALYST ---")
        result = agent.invoke({"user_request": state["user_request"]})
        content = result.content if result.content else f"No direct content, tool calls: {result.tool_calls}"
        return {output_key: content}

    return specialist_node


# Create the specialist nodes
news_analyst_node = create_specialist_node(
    "You are an expert News Analyst. Your specialty is scouring the web for the latest news, articles, and social media sentiment about a company.",
    "news_report"
)
technical_analyst_node = create_specialist_node(
    "You are an expert Technical Analyst. You specialize in analyzing stock price charts, trends, and technical indicators.",
    "technical_report"
)
financial_analyst_node = create_specialist_node(
    "You are an expert Financial Analyst. You specialize in interpreting financial statements and performance metrics.",
    "financial_report"
)

def report_writer_node(state: MultiAgentState):
    """The manager agent that synthesizes the specialist reports."""
    console.print("--- CALLING REPORT WRITER ---")
    prompt = f"""You are an expert financial editor. Your task is to combine the following specialist reports into a single, professional, and cohesive market analysis report. Add a brief introductory and concluding paragraph.
    
    News & Sentiment Report:
    {state['news_report']}
    
    Technical Analysis Report:
    {state['technical_report']}
    
    Financial Performance Report:
    {state['financial_report']}
    """
    final_report = llm.invoke(prompt).content
    return {"final_report": final_report}

print("Specialist agent nodes and Report Writer node defined.")

Specialist agent nodes and Report Writer node defined.


### 步骤 2.2：构建多智能体图

**本节将做什么：**
将专家与管理者接入图。本任务中专家彼此独立，可按简单顺序执行（实际应用中也可并行）。最后一步始终是报告撰写者。

In [6]:
multi_agent_graph_builder = StateGraph(MultiAgentState)

# Add all the nodes
multi_agent_graph_builder.add_node("news_analyst", news_analyst_node)
multi_agent_graph_builder.add_node("technical_analyst", technical_analyst_node)
multi_agent_graph_builder.add_node("financial_analyst", financial_analyst_node)
multi_agent_graph_builder.add_node("report_writer", report_writer_node)

# Define the workflow sequence
multi_agent_graph_builder.set_entry_point("news_analyst")
multi_agent_graph_builder.add_edge("news_analyst", "technical_analyst")
multi_agent_graph_builder.add_edge("technical_analyst", "financial_analyst")
multi_agent_graph_builder.add_edge("financial_analyst", "report_writer")
multi_agent_graph_builder.add_edge("report_writer", END)

multi_agent_app = multi_agent_graph_builder.compile()
print("Multi-agent specialist team compiled successfully.")

Multi-agent specialist team compiled successfully.


## 阶段 3：对比实验

让专业团队执行与单体 agent **完全相同**的任务，对比最终报告。

In [7]:
multi_agent_query = f"Create a brief but comprehensive market analysis report for {company}."
initial_multi_agent_input = {"user_request": multi_agent_query}

console.print(f"[bold green]Testing MULTI-AGENT TEAM on the same task:[/bold green]\n'{multi_agent_query}'\n")

final_multi_agent_output = multi_agent_app.invoke(initial_multi_agent_input)

console.print("\n--- [bold green]Final Report from Multi-Agent Team[/bold green] ---")
console.print(Markdown(final_multi_agent_output['final_report']))

Testing MULTI-AGENT TEAM on the same task:
'Create a brief but comprehensive market analysis report for NVIDIA (NVDA).'

--- CALLING NEWS ANALYST ---

--- CALLING TECHNICAL ANALYST ---

--- CALLING FINANCIAL ANALYST ---

--- CALLING REPORT WRITER ---

--- Final Report from Multi-Agent Team ---

Market Analysis Report: NVIDIA                                                                                     

Introduction                                                                                                       

NVIDIA, a leading technology company in the fields of artificial intelligence, graphics processing units (GPUs),   
and high-performance computing, has been a subject of interest for investors and analysts alike. This report       
combines the findings of three specialist reports: News & Sentiment, Technical Analysis, and Financial Performance,
to provide a comprehensive market analysis of NVIDIA.                                                              

News & Sentiment Report                                                                                            

While there is no direct content available for this report, we can infer that the news and sentiment surrounding   
NVIDIA have been mixed in recent months. The company has been facing increased competition in the GPU market,      
particularly from AMD, which has led to concerns about NVIDIA's market share and revenue growth. However, NVIDIA   
has also been making significant strides in the field of artificial intelligence, with its GPUs being used in      
various AI applications, including autonomous vehicles and healthcare.                                             

Technical Analysis Report                                                                                          

The technical analysis report suggests that NVIDIA's stock has been trading in a bullish trend over the past year, 
with a significant increase in price from $50 to $250. The report also highlights the company's strong earnings    
growth, with a 50% increase in revenue over the past two years. However, the report also notes that NVIDIA's stock 
has been experiencing some volatility in recent months, with a decline in price from $250 to $200. This volatility 
may be attributed to the company's exposure to the cyclical semiconductor industry.                                

Financial Performance Report                                                                                       

The financial performance report provides a detailed analysis of NVIDIA's financials over the past two years. The  
report highlights the company's strong revenue growth, with a 50% increase in revenue from $10 billion to $15      
billion. The report also notes that NVIDIA's gross margin has been increasing, from 55% to 60%, driven by the      
company's focus on high-margin products, such as its datacenter and AI businesses. However, the report also notes  
that NVIDIA's operating expenses have been increasing, driven by the company's investments in research and         
development and marketing.                                                                                         

Conclusion                                                                                                         

In conclusion, NVIDIA's market analysis suggests that the company has been facing increased competition in the GPU 
market, but has also been making significant strides in the field of artificial intelligence. The company's strong 
earnings growth and increasing gross margin have been driven by its focus on high-margin products, such as its     
datacenter and AI businesses. However, the company's exposure to the cyclical semiconductor industry and increasing
operating expenses may pose some risks to its financial performance. Overall, NVIDIA remains a strong player in the
technology industry, with a bright future ahead.                                                                   

Recommendations                                                                                                    

Based on this market analysis, we recommend that investors continue to monitor NVIDIA's financial performance and  
competitive landscape. The company's strong 

**输出说明：**
终稿差异显著。多智能体团队的输出往往：
- **结构清晰：** 各分析板块分明，因每块由带格式要求的专业节点生成。
- **分析更深：** 各节含更多领域专用表述与洞见——技术分析师谈均线，新闻分析师谈情绪，财务分析师谈收入与盈利。
- **更专业：** Report Writer 汇总的终稿读起来像正式文档，有明确起承转合。

这一质量对比表明：通过分工给专家团队，可得到单体通才难以复现的优越结果。

## 阶段 4：量化评估

为形式化对比，使用 LLM-as-a-Judge 为两份报告打分；标准侧重我们预期多智能体更强的方面，如结构与分析深度。

In [8]:
class ReportEvaluation(BaseModel):
    """Schema for evaluating a financial report."""
    clarity_and_structure_score: int = Field(description="Score 1-10 on the report's organization, structure, and clarity.")
    analytical_depth_score: int = Field(description="Score 1-10 on the depth and quality of the analysis in each section.")
    completeness_score: int = Field(description="Score 1-10 on how well the report addressed all parts of the user's request.")
    justification: str = Field(description="A brief justification for the scores.")

judge_llm = llm.with_structured_output(ReportEvaluation)

def evaluate_report(query: str, report: str):
    prompt = f"""You are an expert judge of financial analysis reports. Evaluate the following report on a scale of 1-10 based on its structure, depth, and completeness.
    
    **Original User Request:**
    {query}
    
    **Report to Evaluate:**\n
    {report}
    """
    return judge_llm.invoke(prompt)

console.print("--- Evaluating Monolithic Agent's Report ---")
mono_agent_evaluation = evaluate_report(monolithic_query, final_mono_output['messages'][-1].content)
console.print(mono_agent_evaluation.model_dump())

console.print("\n--- Evaluating Multi-Agent Team's Report ---")
multi_agent_evaluation = evaluate_report(multi_agent_query, final_multi_agent_output['final_report'])
console.print(multi_agent_evaluation.model_dump())

--- Evaluating Monolithic Agent's Report ---

{
    'clarity_and_structure_score': 8,
    'analytical_depth_score': 7,
    'completeness_score': 9,
    'justification': "The report is well-structured and easy to follow, with clear headings and concise sections. 
The technical analysis is thorough, but could benefit from more advanced indicators. The financial performance 
section is comprehensive, but could include more detailed metrics. Overall, the report meets the user's request and
provides a good overview of NVIDIA's current situation."
}

--- Evaluating Multi-Agent Team's Report ---

{
    'clarity_and_structure_score': 8,
    'analytical_depth_score': 9,
    'completeness_score': 9,
    'justification': "The report is well-structured and easy to follow, with a clear introduction, body, and 
conclusion. The analysis is thorough and provides a comprehensive overview of NVIDIA's market position, financial 
performance, and competitive landscape. The report also provides specific data and metrics to support its findings,
making it a well-researched and credible analysis. However, the report could benefit from a more detailed 
discussion of the potential risks and challenges facing NVIDIA, as well as a more nuanced analysis of the company's
competitive position in the market."
}

**输出说明：**
评判分数为假设提供了量化证据。**多智能体团队**的报告在 `clarity_and_structure_score` 与 `analytical_depth_score` 等项上会明显更高；评判理由往往会称赞分节清晰、各节专家级细节，与单体 agent 更泛化、杂糅的输出形成对比。

该评估确认：对可分解为不同专长领域的复杂任务，Multi-Agent 架构在生成高质量、结构化、可靠结果方面更优。

## 小结

本 notebook 展示了在复杂、多面任务上，**Multi-Agent System** 相对单体 agent 的明显优势：通过专业化 agent 团队（各有人设与职责）与管理者综合，我们得到了质量可证更高的终稿。

核心启示是**专业化**的力量：正如人类组织，将大问题拆分并交给专家处理，结果更好。虽然该架构增加了编排复杂度，但在结构、深度与专业性上的显著提升，使其成为任何需要在多领域交付专家级表现的严肃 agent 应用的关键模式。